## Частина 1. Робота з VHI-індексами (NOAA)

### Завдання 1-2: Створення віртуального середовища та автоматизація завантаження файлів за допомогою urllib
**Умова:** Для кожної з адміністративних одиниць України завантажити тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу до його імені додати дату та час завантаження. Реалізувати механізм запобігання повторного довантаження та колізії даних. ID=0 (середнє по Україні) завантажувати не потрібно.

In [1]:
import os
import urllib.request
from datetime import datetime

target_dir = "../datasets/noaa"
os.makedirs(target_dir, exist_ok=True)

def download_vhi_data():
    print(f"--- Старт процесу перевірки та завантаження даних ---")
    
    for province_id in range(1, 28):
        
        already_downloaded = False
        for file in os.listdir(target_dir):
            if file.startswith(f"vhi_id_{province_id}_") and file.endswith(".csv"):
                print(f"Область №{province_id} вже завантажена раніше (файл: {file}). Пропускаємо.")
                already_downloaded = True
                break
                
        if already_downloaded:
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{target_dir}/vhi_id_{province_id}_{timestamp}.csv"
            
            urllib.request.urlretrieve(url, filename)
            print(f"Успішно завантажено область №{province_id} -> {filename}")
        except Exception as e:
            print(f"Помилка при завантаженні області №{province_id}: {e}")
            
    print(f"--- Процес завершено ---")

download_vhi_data()

--- Старт процесу перевірки та завантаження даних ---
Успішно завантажено область №1 -> ../datasets/noaa/vhi_id_1_20260520_025108.csv
Успішно завантажено область №2 -> ../datasets/noaa/vhi_id_2_20260520_025123.csv
Успішно завантажено область №3 -> ../datasets/noaa/vhi_id_3_20260520_025129.csv
Успішно завантажено область №4 -> ../datasets/noaa/vhi_id_4_20260520_025132.csv
Успішно завантажено область №5 -> ../datasets/noaa/vhi_id_5_20260520_025142.csv
Успішно завантажено область №6 -> ../datasets/noaa/vhi_id_6_20260520_025143.csv
Успішно завантажено область №7 -> ../datasets/noaa/vhi_id_7_20260520_025153.csv
Успішно завантажено область №8 -> ../datasets/noaa/vhi_id_8_20260520_025154.csv
Успішно завантажено область №9 -> ../datasets/noaa/vhi_id_9_20260520_025155.csv
Успішно завантажено область №10 -> ../datasets/noaa/vhi_id_10_20260520_025157.csv
Успішно завантажено область №11 -> ../datasets/noaa/vhi_id_11_20260520_025158.csv
Успішно завантажено область №12 -> ../datasets/noaa/vhi_id_12_

### Завдання 3-4: Зчитування даних у Pandas DataFrame, Data Cleaning та реіндексація за українським алфавітом
**Умова:** Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області. Реалізувати процедуру зміни індексів: замінити індекси з NOAA (англійська абетка) так, щоб області індексувалися за українською абеткою (1 область - Вінницька).

In [18]:
import pandas as pd
import glob
import re

clean_nav_dict = {
    1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька",
    5: "Житомирська", 6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська",
    9: "Київська", 10: "Кіровоградська", 11: "Луганська", 12: "Львівська",
    13: "Миколаївська", 14: "Одеська", 15: "Полтавська", 16: "Рівненська",
    17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
    21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська",
    25: "Республіка Крим", 26: "м. Севастополь", 27: "м. Київ"
}

noaa_to_ua_id = {
    1: 22, 2: 24, 3: 23, 4: 3, 5: 4, 6: 20, 7: 21, 8: 8, 9: 9, 10: 10,
    11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18,
    19: 6, 20: 19, 21: 20, 22: 1, 23: 2, 24: 5, 25: 7, 26: 25, 27: 26
}

def load_and_clean_data(folder_path="../datasets/noaa"):
    all_frames = []
    file_paths = glob.glob(f"{folder_path}/vhi_id_*.csv")
    
    if not file_paths:
        print("Помилка: не знайдено CSV-файлів у папці ../datasets/noaa")
        return pd.DataFrame()
        
    for path in file_paths:
        match = re.search(r"vhi_id_(\d+)_", path)
        if not match:
            continue
        noaa_id = int(match.group(1))
        
        ua_id = noaa_to_ua_id.get(noaa_id, noaa_id)
        region_name = clean_nav_dict.get(ua_id, "Невідома область")
        
        try:
            df = pd.read_csv(
                path, 
                sep=r'[\s,]+', 
                engine='python',
                comment='<',
                header=None
            )
        except Exception as e:
            print(f"Не вдалося прочитати файл {path}: {e}")
            continue
            
        if df.empty:
            continue
            
        df[0] = df[0].astype(str).str.strip()
        df = df[df[0].str.isnumeric() == True]
        
        if df.shape[1] < 7:
            continue
            
        df = df.iloc[:, :7]
        df.columns = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
        
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
        df = df.dropna(subset=['Year', 'Week', 'VHI'])
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        
        df = df[df['VHI'] >= 0]
        
        if not df.empty:
            df['Area_ID'] = ua_id
            df['Area_Name'] = region_name
            
            df = df[['Area_ID', 'Area_Name', 'Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]
            all_frames.append(df)
            
    if not all_frames:
        print("Помилка: Не вдалося розпарсити жодного файлу. Перевір вміст папки datasets.")
        return pd.DataFrame()
        
    main_df = pd.concat(all_frames, ignore_index=True)
    main_df = main_df.sort_values(by=['Area_ID', 'Year', 'Week']).reset_index(drop=True)
    return main_df

vhi_df = load_and_clean_data()
print("--- Дані УСПІШНО оброблені та завантажені! ---")
print(f"Загальна кількість записів у таблиці: {vhi_df.shape[0]}")
vhi_df.head(15)

--- Дані УСПІШНО оброблені та завантажені! ---
Загальна кількість записів у таблиці: 58995


,Area_ID,Area_Name,Year,Week,SMN,SMT,VCI,TCI,VHI
0,1,Вінницька,1982,2,0.047,258.77,42.75,49.09,45.92
1,1,Вінницька,1982,3,0.045,260.25,43.44,43.56,43.50
2,1,Вінницька,1982,4,0.040,261.37,37.43,40.81,39.12
3,1,Вінницька,1982,5,0.037,262.61,30.04,41.17,35.60
4,1,Вінницька,1982,6,0.035,263.47,24.65,42.49,33.57
5,1,Вінницька,1982,7,0.035,264.14,22.80,43.61,33.20
6,1,Вінницька,1982,8,0.037,265.79,22.50,42.95,32.73
7,1,Вінницька,1982,9,0.038,267.78,21.73,43.87,32.80
8,1,Вінницька,1982,10,0.041,270.26,20.93,43.98,32.46
9,1,Вінницька,1982,11,0.049,273.33,21.53,40.51,31.02


### Завдання 5: Процедури пошуку екстремумів та фільтрації даних
**Умова:** Реалізувати наступні функції для роботи з отриманим DataFrame:
1. Ряд VHI для заданого року та заданої області.
2. Ряд VHI за вказаний діапазон років для вказаних областей.
3. Пошук екстремумів (мінімум та максимум) VHI для заданого року та області.

In [12]:
def get_vhi_series(df, year, area_id):
    """Повертає фільтрований DataFrame або Series із тижнями та значеннями VHI"""
    result = df.loc[(df['Year'] == year) & (df['Area_ID'] == area_id), ['Week', 'VHI']]
    return result.reset_index(drop=True)
    
print("1. Тест вибірки ряду VHI для області №1 за 2020 рік (перші 5 рядків):")
display(get_vhi_series(vhi_df, year=2020, area_id=1))

1. Тест вибірки ряду VHI для області №1 за 2020 рік (перші 5 рядків):


,Week,VHI
0,1,41.75
1,2,43.90
2,3,45.12
3,4,45.32
4,5,44.78
5,6,43.67
6,7,43.46
7,8,44.86
8,9,46.62
9,10,48.32


In [7]:
def get_vhi_time_range(df, start_year, end_year, area_ids):
    result = df[(df['Year'] >= start_year) & 
                (df['Year'] <= end_year) & 
                (df['Area_ID'].isin(area_ids))]
    
    return result.reset_index(drop=True)

print("--- Завдання: Ряд VHI за діапазон років для вказаних областей ---")
historical_data = get_vhi_time_range(vhi_df, start_year=2015, end_year=2020, area_ids=[1, 2])

display(historical_data)

--- Завдання: Ряд VHI за діапазон років для вказаних областей ---


,Area_ID,Area_Name,Year,Week,VHI
0,1,Вінницька,2015,1,47.82
1,1,Вінницька,2015,2,50.16
2,1,Вінницька,2015,3,50.53
3,1,Вінницька,2015,4,49.33
4,1,Вінницька,2015,5,46.84
...,...,...,...,...,...
619,2,Волинська,2020,48,43.67
620,2,Волинська,2020,49,44.16
621,2,Волинська,2020,50,44.15
622,2,Волинська,2020,51,43.84


In [3]:
def calculate_statistics(df, year, area_id):
    sub_df = df[(df['Year'] == year) & (df['Area_ID'] == area_id)]
    if sub_df.empty:
        print(f"Дані для Області {area_id} за {year} рік відсутні.")
        return
    
    vhi_series = sub_df['VHI']
    
    print(f"--- Статистика VHI для області №{area_id} ({sub_df['Area_Name'].iloc[0]}) за {year} рік ---")
    print(f"Мінімум VHI: {vhi_series.min()}")
    print(f"Максимум VHI: {vhi_series.max()}")
    print(f"Середнє значення VHI: {vhi_series.mean():.4f}")
    print(f"Медіана VHI: {vhi_series.median():.4f}")
    print("-" * 50)

calculate_statistics(vhi_df, year=2020, area_id=1)

--- Статистика VHI для області №1 (Вінницька) за 2020 рік ---
Мінімум VHI: 40.2
Максимум VHI: 68.55
Середнє значення VHI: 51.2488
Медіана VHI: 46.6500
--------------------------------------------------
